<a href="https://colab.research.google.com/github/zhaoyingjun/Tiny-R2/blob/main/Tiny_R2_journey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tiktoken datasets transformers huggingface_hub
!pip install git+https://github.com/KellerJordan/Muon
!pip install --upgrade transformers

In [ ]:
!hf auth login --force

In [2]:
!git clone https://github.com/zhaoyingjun/Tiny-R2.git
%cd Tiny-R2

Cloning into 'Tiny-R2'...
remote: Enumerating objects: 282, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 282 (delta 65), reused 0 (delta 0), pack-reused 166 (from 1)
Receiving objects: 100% (282/282), 2.15 MiB | 7.64 MiB/s, done.
Resolving deltas: 100% (158/158), done.
/content/Tiny-R2


**训练openwebtext数据，可以训练一个基础大模型，1B参数量，评测模型结构和效果。**

In [ ]:
!python train.py --n_layer 6 --n_embd 768 --hc 'True' --mhc 'True' --n_experts 32  --max_iters 10000 --attention_types 'Sparse' --batch_size 8 --ctx_len 2048 --hf_dataset 'karpathy/climbmix-400b-shuffle' --resume True --save_best_only True

预训练完成之后，进行benchmark的验证

In [ ]:
!python /content/Tiny-R2/evaluate.py --checkpoint /content/Tiny-R2/checkpoints/best_model_step_3980.pt

直接使用Qwen3.5-9模型作为教师模式进行OPD训练，采用数据集GPQA

In [ ]:
!python opd_train.py --batch_size 2 --ctx_len 2048 --hf_teacher_model Qwen/Qwen3.5-9B --student_ckpt "./checkpoints/best_model_step_0.pt" --tokenizer_name Qwen/Qwen3.5-9B

✅ 成功导入 Tiny-R2 核心模块 (model, config)
✅ 分布式环境初始化成功（单卡模式）
📝 加载 Tokenizer
✅ 词汇量: 248077

📊 加载 GPQA 子集: gpqa_diamond
📊 数据集加载完成 | 样本数: 159
📊 数据集加载完成 | 样本数: 39

🤖 初始化 Tiny-R2 模型
num Muon parameters: 929,363,376
num AdamW parameters: 272,371,392
📂 加载检查点: ./checkpoints/best_model_step_0.pt
⚠️  词汇表对齐: token_embedding_table.weight | 检查点 248044 -> 当前 248077
⚠️  词汇表对齐: lm_head.weight | 检查点 248044 -> 当前 248077
✅ 模型权重加载成功！
ℹ️  优化器状态将重新初始化（避免维度不匹配）
✅ 调度器状态加载成功
✅ 检查点加载完成 | 恢复迭代: 0 | 最优val_loss: 40.1264

👨‍🏫 加载教师模型 Qwen3.5-9B
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 4 files: 100% 4/4 [00:00<00:00, 98689.51it/s]
Download complete: : 0.00B [00:00, ?B/s]              [transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Download complete: : 0.00B [00:00, ?B/s]
Loadi

In [ ]:
!python /content/Tiny-R2/sampler.py --checkpoint /content/Tiny-R2/checkpoints/best_model_step_3980.pt --prompt "Hi my name is enjoy . "  --max_new_tokens 100 --temperature 0.7    --top_k 100     --device cuda